# 06 — Indeks spasial H3

Kelurahan sangat beragam ukurannya, sehingga membandingkan jumlah antar-kelurahan menyesatkan. H3 membagi
ruang menjadi heksagon seragam pada resolusi tertentu — setiap angka punya penyebut yang sama.

In [ ]:
import sys
sys.path.insert(0, "../scripts")

import geopandas as gpd
import h3
import pandas as pd
from shapely.geometry import Polygon

from kota import CFG, PROC, UTM

kel = gpd.read_file(PROC / "morfologi.gpkg", layer="kelurahan")
bangunan = gpd.read_file(PROC / "bangunan_utm.gpkg", layer="bangunan")

## 1. Masalahnya: unit administratif tidak seragam

In [ ]:
print(f"Kelurahan terkecil {kel.luas_ha.min():.0f} ha, terbesar {kel.luas_ha.max():.0f} ha "
      f"— {kel.luas_ha.max() / kel.luas_ha.min():.0f} kali lipat")
kel.sort_values("luas_ha").iloc[[0, -1]][["kelurahan", "kecamatan", "luas_ha", "n_bangunan"]].round(1)

## 2. Dasar H3
Satu titik → satu sel, pada resolusi 0 (terkasar) hingga 15 (terhalus).

In [ ]:
lat, lon = CFG["lat"], CFG["lon"]
for res in [7, 8, 9, 10]:
    print(f"res {res:2}: {h3.latlng_to_cell(lat, lon, res)}  rerata luas {h3.average_hexagon_area(res, 'km^2'):.4f} km²")

## 3. Agregasi bangunan ke heksagon

In [ ]:
pt = bangunan.representative_point().to_crs(4326)
bangunan["lat"], bangunan["lon"] = pt.y, pt.x


def hex_gdf(res):
    sel = [h3.latlng_to_cell(a, b, res) for a, b in zip(bangunan["lat"], bangunan["lon"])]
    g = bangunan.assign(h3=sel).groupby("h3").agg(n_bangunan=("tapak_m2", "size"), tapak_m2=("tapak_m2", "sum"))
    geom = [Polygon([(x, y) for y, x in h3.cell_to_boundary(c)]) for c in g.index]
    g = gpd.GeoDataFrame(g.reset_index(), geometry=geom, crs=4326).to_crs(UTM)
    g["luas_ha"] = g.area / 1e4
    g["bangunan_per_ha"] = g["n_bangunan"] / g["luas_ha"]
    g["rasio_tutupan"] = g["tapak_m2"] / (g["luas_ha"] * 1e4)
    g["res"] = res
    return g


hexes = {res: hex_gdf(res) for res in [7, 8, 9]}
pd.DataFrame({res: {"jumlah sel": len(g), "luas sel (ha)": g.luas_ha.median(),
                    "maks bangunan/ha": g.bangunan_per_ha.max()} for res, g in hexes.items()}).round(1)

## 4. Mengapa heksagon
Keenam tetangga sebuah heksagon berjarak *hampir* sama dari pusatnya. Hampir, karena heksagon H3 dibentuk
di atas ikosahedron dan sedikit terdistorsi — selisihnya beberapa persen. Pada grid bujur sangkar, tetangga
diagonal 41 % lebih jauh dari tetangga sisi: bias bagi analisis apa pun yang melibatkan ketetanggaan.

In [ ]:
c = h3.latlng_to_cell(lat, lon, 8)
pusat = h3.cell_to_latlng(c)
from pyproj import Geod
geod = Geod(ellps="WGS84")
jarak = [geod.inv(pusat[1], pusat[0], h3.cell_to_latlng(n)[1], h3.cell_to_latlng(n)[0])[2]
         for n in h3.grid_ring(c, 1)]
print("jarak ke 6 tetangga (m):", [round(d) for d in jarak])
print(f"terjauh / terdekat: {max(jarak) / min(jarak):.2f}   (bujur sangkar: 1.41)")

## 5. Resolusi mengubah cerita
Data yang sama, tiga peta. Pilih resolusi dengan sengaja, dan selalu sebutkan resolusinya.

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (res, g) in zip(axes, hexes.items()):
    g.plot(column="bangunan_per_ha", cmap="magma_r", ax=ax, vmin=0, vmax=60, linewidth=0)
    ax.set_title(f"H3 res {res} — {len(g)} sel")
    ax.set_axis_off()

In [ ]:
pd.concat(hexes.values()).to_file(PROC / "h3_bangunan.gpkg", layer="h3")
print("disimpan: h3_bangunan.gpkg")